# Historical drought exposure map

Pick a **state**, **crop**, **drought definition** and **map measure**; the county map updates.

**What this map shows.** It describes what *already happened* in 2000–2025. It is **historical exposure**, not a forecast and not a risk estimate ("at risk").

**Drought definitions.** `max_drought_severity` in the master dataset is the worst U.S. Drought Monitor category reached in any week of the growing season (more than 1% of the county affected). The scale is offset by 1 from USDM's own labels:

| Definition | `max_drought_severity` | USDM label |
|---|---|---|
| D1+ moderate or worse | ≥ 2 | D1–D4 |
| D2+ severe or worse | ≥ 3 | D2–D4 |
| D3+ extreme or worse | ≥ 4 | D3–D4 |
| D4 exceptional | = 5 | D4 |

A county-year counts as a **drought year** when its season maximum reaches the chosen level.

**Map measures.**
- *Drought exposure*: number of drought years per county (hover shows the share of observed years).
- *Priority score (top 20% outlined)*: expected bushels lost per year = how often the county hit the chosen drought level × its typical yield loss in those years × its typical harvested acres × its typical yield. Loss is pulled toward the state-crop average when a county has few drought years, so one bad year cannot put a county at the top. The 20% of counties with the highest score are outlined in black and listed in a table under the map. This ranks **past drought exposure and losses**. It does not use soil storage, because the soil comparison below finds no consistent protective effect, and it is not a forecast.
- *Drought-year losses*: mean `yield_anomaly_pct` over that county's drought years that have a reported yield. Negative = below the county's own trend. Counties with no drought year that has a yield are left blank (blank is not zero).


In [1]:
import sys
sys.path.insert(0, "..")   # this notebook lives in notebooks/, src/ is one level up

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

from src.data_loader import load_master, load_counties

df = load_master()
counties = load_counties()   # GeoJSON, feature id = fips
print(f"{len(df):,} rows loaded")

17,056 rows loaded


In [2]:
# Dropdown options: label shown -> value used in the data
STATES = {
    "Iowa": "IA",
    "Nebraska": "NE",
    "California": "CA",
    "Delaware": "DE",
}
CROPS = {
    "Corn (grain)": "CORN",
    "Soybeans": "SOYBEANS",
    "Winter wheat": "WHEAT",
    "Grain sorghum": "SORGHUM",
}
# max_drought_severity is offset by 1 from USDM labels: 2 = D1, 3 = D2, 4 = D3, 5 = D4
DROUGHT_DEFS = {
    "D1+ moderate or worse (severity ≥ 2)": 2,
    "D2+ severe or worse (severity ≥ 3)": 3,
    "D3+ extreme or worse (severity ≥ 4)": 4,
    "D4 exceptional (severity = 5)": 5,
}
MEASURES = {
    "Drought exposure (number of drought years)": "exposure",
    "Drought-year losses (mean yield anomaly)": "loss",
    "Priority score (expected bushels lost per year)": "priority",
}

In [3]:
def county_table(state, crop, threshold):
    """One row per county for the chosen state, crop and drought definition."""
    sub = df[(df["state_alpha"] == state) & (df["crop"] == crop)].copy()

    # Drop rows without drought data (e.g. wheat 2000, whose season starts before the
    # Drought Monitor archive does). NaN is not "no drought".
    sub = sub[sub["max_drought_severity"].notna()]
    sub["is_drought"] = sub["max_drought_severity"] >= threshold

    # Exposure: how many seasons reached the chosen drought level
    out = (
        sub.groupby(["fips", "county_name"], as_index=False)
        .agg(years_observed=("year", "nunique"),
             drought_years=("is_drought", "sum"))
    )
    out["pct_years_in_drought"] = 100 * out["drought_years"] / out["years_observed"]

    # Losses: yield anomaly in those drought years, only where a yield (and trend) exists
    dy = sub[sub["is_drought"] & sub["yield_anomaly_pct"].notna()]
    loss = (
        dy.groupby("fips", as_index=False)
        .agg(mean_yield_anomaly_pct=("yield_anomaly_pct", "mean"),
             worst_yield_anomaly_pct=("yield_anomaly_pct", "min"),
             drought_years_with_yield=("year", "nunique"))
    )
    return out.merge(loss, on="fips", how="left")


PRIORITY_SHARE = 0.20   # top 20% of counties are flagged as priority
SHRINK_K = 3            # a county needs about 3 drought years before its own loss counts as much as the state average

def build_priority(state, crop, threshold, years=None):
    """Historical priority score per county: expected bushels lost per year.

    score = (share of years at the drought level)
            x (shrunk mean loss in drought years, as a fraction)
            x (mean yield, bu/acre) x (mean harvested acres)
    Past exposure and past losses only. It does not use soil.
    """
    d = df[(df["state_alpha"] == state) & (df["crop"] == crop) & df["max_drought_severity"].notna()].copy()
    if years is not None:
        d = d[d["year"].isin(years)]
    d["is_drought"] = d["max_drought_severity"] >= threshold

    g = (
        d.groupby(["fips", "county_name"], as_index=False)
        .agg(years_observed=("year", "nunique"),
             drought_years=("is_drought", "sum"),
             avg_yield=("yield_per_acre", "mean"),
             avg_acres=("acres_harvested", "mean"))
    )

    dy = d[d["is_drought"] & d["yield_anomaly_pct"].notna()]
    loss = (
        dy.groupby("fips", as_index=False)
        .agg(raw_loss_pct=("yield_anomaly_pct", "mean"),
             drought_years_with_yield=("year", "nunique"))
    )
    g = g.merge(loss, on="fips", how="left")
    g["drought_years_with_yield"] = g["drought_years_with_yield"].fillna(0)

    # Pull thin counties toward the state-crop average drought-year anomaly
    prior = dy["yield_anomaly_pct"].mean() if len(dy) else 0.0
    n = g["drought_years_with_yield"]
    g["loss_pct"] = (n * g["raw_loss_pct"].fillna(0) + SHRINK_K * prior) / (n + SHRINK_K)

    g["freq_drought"] = g["drought_years"] / g["years_observed"]
    g["loss_frac"] = (-g["loss_pct"]).clip(lower=0) / 100       # only losses count; gains count as 0
    g["expected_loss_bu"] = g["freq_drought"] * g["loss_frac"] * g["avg_yield"] * g["avg_acres"]

    scored = g["expected_loss_bu"].notna()
    g["priority_rank"] = g.loc[scored, "expected_loss_bu"].rank(ascending=False, method="min")
    n_flag = max(1, int(round(PRIORITY_SHARE * scored.sum()))) if scored.any() else 0
    g["priority_flag"] = g["priority_rank"] <= n_flag
    return g.sort_values("priority_rank")


In [4]:
def make_map(state_label, crop_label, def_label, measure):
    """Returns (figure or None, note, top-counties table or None)."""
    state, crop, threshold = STATES[state_label], CROPS[crop_label], DROUGHT_DEFS[def_label]
    tbl = county_table(state, crop, threshold)

    if tbl.empty:
        return None, f"No data for {crop_label} in {state_label} (that crop is not grown there in this dataset).", None

    base_hover = {
        "fips": False,
        "drought_years": True,
        "years_observed": True,
        "pct_years_in_drought": ":.0f",
        "drought_years_with_yield": True,
        "mean_yield_anomaly_pct": ":.1f",
    }
    labels = {
        "drought_years": "Drought years",
        "years_observed": "Years observed",
        "pct_years_in_drought": "% of years in drought",
        "drought_years_with_yield": "Drought years with yield",
        "mean_yield_anomaly_pct": "Mean yield anomaly (%)",
    }
    title_core = f"{state_label} {crop_label}: {def_label}, 2000\u20132025"
    top_table = None

    if measure == "exposure":
        plot_df = tbl
        fig = px.choropleth(
            plot_df, geojson=counties, locations="fips",
            color="drought_years", color_continuous_scale="YlOrRd",
            range_color=(0, max(1, int(plot_df["drought_years"].max()))),
            hover_name="county_name", hover_data=base_hover, labels=labels,
            title=f"Historical drought exposure<br><sup>{title_core}</sup>",
        )
        fig.update_layout(coloraxis_colorbar_title="Drought<br>years")
        note = f"{len(plot_df)} counties shown. Counts seasons that reached the chosen drought level."

    elif measure == "loss":
        plot_df = tbl.dropna(subset=["mean_yield_anomaly_pct"])
        if plot_df.empty:
            return None, ("No county has a drought year with a reported yield under this definition. "
                          "Try a milder drought definition."), None
        fig = px.choropleth(
            plot_df, geojson=counties, locations="fips",
            color="mean_yield_anomaly_pct", color_continuous_scale="RdYlGn",
            range_color=(-40, 40),
            hover_name="county_name", hover_data=base_hover, labels=labels,
            title=f"Historical drought-year losses<br><sup>{title_core}</sup>",
        )
        fig.update_layout(coloraxis_colorbar_title="Mean yield<br>anomaly (%)")
        thin = int((plot_df["drought_years_with_yield"] <= 2).sum())
        note = (f"{len(plot_df)} of {len(tbl)} counties shown (blank = no drought year with a reported yield, "
                f"not zero loss). {thin} shown counties rest on 1-2 drought years, so treat them as noisy.")

    else:  # priority
        pr = build_priority(state, crop, threshold)
        plot_df = pr.dropna(subset=["expected_loss_bu"])
        if plot_df.empty:
            return None, ("No county has both a drought year with a reported yield and reported acres under "
                          "this definition. Try a milder drought definition."), None
        fig = px.choropleth(
            plot_df, geojson=counties, locations="fips",
            color="expected_loss_bu", color_continuous_scale="OrRd",
            hover_name="county_name",
            hover_data={
                "fips": False,
                "priority_rank": ":.0f",
                "expected_loss_bu": ":,.0f",
                "drought_years": True,
                "years_observed": True,
                "loss_pct": ":.1f",
                "avg_acres": ":,.0f",
            },
            labels={
                "priority_rank": "Priority rank",
                "expected_loss_bu": "Expected bushels lost / year",
                "drought_years": "Drought years",
                "years_observed": "Years observed",
                "loss_pct": "Typical drought-year yield anomaly (%)",
                "avg_acres": "Avg harvested acres",
            },
            title=f"Historical drought priority (past exposure \u00d7 past loss \u00d7 acres)<br><sup>{title_core}</sup>",
        )
        fig.update_layout(coloraxis_colorbar_title="Expected<br>bushels lost<br>per year")

        flagged = plot_df[plot_df["priority_flag"]]
        fig.add_trace(go.Choropleth(
            geojson=counties, locations=flagged["fips"], z=[1] * len(flagged),
            colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]], showscale=False,
            marker_line_color="black", marker_line_width=2.5,
            hoverinfo="skip", name=f"Priority: top {int(PRIORITY_SHARE * 100)}%", showlegend=True,
        ))
        fig.update_layout(legend=dict(orientation="h", y=-0.02))

        top_table = (
            flagged[["priority_rank", "county_name", "expected_loss_bu", "drought_years",
                     "years_observed", "loss_pct", "avg_acres"]]
            .rename(columns={
                "priority_rank": "Rank", "county_name": "County",
                "expected_loss_bu": "Expected bu lost / yr", "drought_years": "Drought years",
                "years_observed": "Years observed", "loss_pct": "Typical drought-yr anomaly (%)",
                "avg_acres": "Avg harvested acres"})
            .round({"Expected bu lost / yr": 0, "Typical drought-yr anomaly (%)": 1, "Avg harvested acres": 0})
            .astype({"Rank": int})
            .reset_index(drop=True)
        )
        note = (f"{len(flagged)} of {len(plot_df)} scored counties outlined as priority "
                f"({len(tbl) - len(plot_df)} counties blank: no reported yield or acres, not zero). "
                "Score = drought frequency x typical loss x yield x acres; loss is shrunk toward the state average "
                "for counties with few drought years. Soil is not used.")

    fig.update_geos(fitbounds="locations", visible=False)
    fig.update_layout(margin=dict(l=0, r=0, t=70, b=0), height=560)
    return fig, note, top_table


In [5]:
state_dd   = widgets.Dropdown(options=list(STATES),       value="Iowa",  description="State:",   layout=widgets.Layout(width="260px"))
crop_dd    = widgets.Dropdown(options=list(CROPS),        value="Corn (grain)", description="Crop:", layout=widgets.Layout(width="260px"))
def_dd     = widgets.Dropdown(options=list(DROUGHT_DEFS), value="D2+ severe or worse (severity ≥ 3)",
                              description="Drought:", layout=widgets.Layout(width="420px"))
measure_dd = widgets.Dropdown(options=[(k, v) for k, v in MEASURES.items()], value="exposure",
                              description="Map shows:", layout=widgets.Layout(width="420px"))
out = widgets.Output()

def refresh(_=None):
    with out:
        out.clear_output(wait=True)
        fig, note, top_table = make_map(state_dd.value, crop_dd.value, def_dd.value, measure_dd.value)
        if fig is not None:
            fig.show()
        print(note)
        print("Historical exposure, 2000\u20132025. Not a forecast; not a risk estimate.")
        if top_table is not None:
            display(top_table)

for w in (state_dd, crop_dd, def_dd, measure_dd):
    w.observe(refresh, names="value")

display(widgets.VBox([widgets.HBox([state_dd, crop_dd]), widgets.HBox([def_dd, measure_dd]), out]))
refresh()

# Evidence Found

We found no consistent evidence that higher soil water storage is associated with smaller yield losses during D2+ drought across the analyzed state-crop combinations.

The following code demonstrates that.

In [6]:
print("Total rows:", len(df))

print("\nUsable yield anomaly:")
print(df["yield_anomaly_pct"].notna().sum())

print("\nMissing yield anomaly:")
print(df["yield_anomaly_pct"].isna().sum())

print("\nUsable rows by state:")
print(
    df.groupby("state_alpha")["yield_anomaly_pct"]
      .apply(lambda x: x.notna().sum())
)

print("\nUsable rows by state and crop:")
print(
    df.groupby(["state_alpha", "crop"])["yield_anomaly_pct"]
      .apply(lambda x: x.notna().sum())
)

Total rows: 17056

Usable yield anomaly:
10995

Missing yield anomaly:
6061

Usable rows by state:
state_alpha
CA     514
DE     180
IA    4907
NE    5394
Name: yield_anomaly_pct, dtype: int64

Usable rows by state and crop:
state_alpha  crop    
CA           CORN         201
             SORGHUM        0
             WHEAT        313
DE           CORN          78
             SORGHUM        0
             SOYBEANS      72
             WHEAT         30
IA           CORN        2470
             SOYBEANS    2437
             WHEAT          0
NE           CORN        2058
             SORGHUM      433
             SOYBEANS    1709
             WHEAT       1194
Name: yield_anomaly_pct, dtype: int64


This data is important becuase not all columns have data (NaN), despite this, they are not zero. Therefore, they should still be considered.

The following table is a breakdown of the four states and their respective usable rows per crop.

In [7]:
analysis_df = df[
    df["yield_anomaly_pct"].notna() &
    (df["max_drought_severity"] > 3)
].copy()

print("\nDrought-year observations by state and crop:")
print(analysis_df.groupby(["state_alpha", "crop"]).size())


Drought-year observations by state and crop:
state_alpha  crop    
CA           CORN         29
             WHEAT        67
DE           CORN          3
             SOYBEANS      3
             WHEAT         5
IA           CORN        277
             SOYBEANS    317
NE           CORN        471
             SORGHUM     114
             SOYBEANS    397
             WHEAT       316
dtype: int64


In [8]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# STEP 1: Start with usable drought-year observations
# ---------------------------------------------------------
analysis_df = df[
    df["yield_anomaly_pct"].notna() &
    (df["max_drought_severity"] >= 3) &
    df["aws_100cm_mm"].notna()
].copy()


# ---------------------------------------------------------
# STEP 2: Get ONE soil-storage value per county/crop
# ---------------------------------------------------------
county_soil = (
    df[
        df["aws_100cm_mm"].notna()
    ][
        [
            "state_alpha",
            "crop",
            "fips",
            "county_name",
            "aws_100cm_mm"
        ]
    ]
    .drop_duplicates(
        subset=["state_alpha", "crop", "fips"]
    )
)


# ---------------------------------------------------------
# STEP 3: Find the median soil storage within each
# state + crop combination
# ---------------------------------------------------------
county_soil["soil_median"] = (
    county_soil
    .groupby(["state_alpha", "crop"])["aws_100cm_mm"]
    .transform("median")
)


# ---------------------------------------------------------
# STEP 4: Divide counties into High vs Low soil storage
# ---------------------------------------------------------
county_soil["soil_group"] = np.where(
    county_soil["aws_100cm_mm"] >= county_soil["soil_median"],
    "High",
    "Low"
)


# ---------------------------------------------------------
# STEP 5: Attach ONLY the soil group to drought data
# ---------------------------------------------------------
analysis_df = analysis_df.merge(
    county_soil[
        [
            "state_alpha",
            "crop",
            "fips",
            "soil_group"
        ]
    ],
    on=["state_alpha", "crop", "fips"],
    how="inner"
)


# ---------------------------------------------------------
# STEP 6: Average each county's yield anomaly across
# its D2+ drought years
# ---------------------------------------------------------
county_drought = (
    analysis_df
    .groupby(
        [
            "state_alpha",
            "crop",
            "fips",
            "soil_group"
        ],
        as_index=False
    )
    .agg(
        aws_100cm_mm=("aws_100cm_mm", "first"),
        drought_yield_anomaly=("yield_anomaly_pct", "mean"),
        drought_years=("year", "nunique")
    )
)


# ---------------------------------------------------------
# STEP 7: Count counties in each state/crop
# ---------------------------------------------------------
county_counts = (
    county_drought
    .groupby(["state_alpha", "crop"])["fips"]
    .nunique()
    .reset_index(name="county_count")
)

valid_combinations = county_counts[
    county_counts["county_count"] >= 10
][
    ["state_alpha", "crop"]
]


county_drought = county_drought.merge(
    valid_combinations,
    on=["state_alpha", "crop"],
    how="inner"
)


# ---------------------------------------------------------
# STEP 8: Calculate average drought loss for High vs Low
# ---------------------------------------------------------
results = (
    county_drought
    .groupby(
        ["state_alpha", "crop", "soil_group"]
    )
    .agg(
        counties=("fips", "nunique"),
        avg_drought_yield_anomaly=(
            "drought_yield_anomaly",
            "mean"
        )
    )
    .reset_index()
)


# ---------------------------------------------------------
# STEP 9: Put High and Low side-by-side
# ---------------------------------------------------------
comparison = results.pivot_table(
    index=["state_alpha", "crop"],
    columns="soil_group",
    values=[
        "counties",
        "avg_drought_yield_anomaly"
    ]
)

comparison.columns = [
    f"{col[1].lower()}_{col[0]}"
    for col in comparison.columns
]

comparison = comparison.reset_index()


# ---------------------------------------------------------
# STEP 10: Calculate High - Low
# ---------------------------------------------------------
comparison["high_minus_low"] = (
    comparison["high_avg_drought_yield_anomaly"]
    -
    comparison["low_avg_drought_yield_anomaly"]
)


print("HIGH vs LOW SOIL STORAGE DURING D2+ DROUGHT")
print("=" * 70)
print(comparison.round(2).to_string(index=False))

HIGH vs LOW SOIL STORAGE DURING D2+ DROUGHT
state_alpha     crop  high_avg_drought_yield_anomaly  low_avg_drought_yield_anomaly  high_counties  low_counties  high_minus_low
         CA     CORN                           -1.52                           2.34            7.0           6.0           -3.86
         CA    WHEAT                            2.27                           0.73           11.0           8.0            1.54
         IA     CORN                           -6.39                          -5.68           50.0          49.0           -0.71
         IA SOYBEANS                           -5.40                          -6.11           50.0          49.0            0.71
         NE     CORN                           -7.90                          -3.78           46.0          43.0           -4.12
         NE  SORGHUM                          -12.47                         -14.15           22.0           8.0            1.68
         NE SOYBEANS                           -7.21 

The following code then provides the bootstrap estimates:

In [9]:
import numpy as np

def bootstrap_difference(data, n_boot=2000, random_state=42):
    rng = np.random.default_rng(random_state)

    high = data[data["soil_group"] == "High"]["drought_yield_anomaly"].dropna().values
    low = data[data["soil_group"] == "Low"]["drought_yield_anomaly"].dropna().values

    if len(high) < 2 or len(low) < 2:
        return np.nan, np.nan, np.nan

    differences = []

    for _ in range(n_boot):
        high_sample = rng.choice(
            high,
            size=len(high),
            replace=True
        )

        low_sample = rng.choice(
            low,
            size=len(low),
            replace=True
        )

        differences.append(
            high_sample.mean() - low_sample.mean()
        )

    differences = np.array(differences)

    return (
        differences.mean(),
        np.percentile(differences, 2.5),
        np.percentile(differences, 97.5)
    )


bootstrap_results = []

for (state, crop), group in county_drought.groupby(
    ["state_alpha", "crop"]
):
    estimate, lower, upper = bootstrap_difference(group)

    bootstrap_results.append({
        "state_alpha": state,
        "crop": crop,
        "estimate": estimate,
        "lower_95": lower,
        "upper_95": upper
    })


bootstrap_results = pd.DataFrame(bootstrap_results)

print("BOOTSTRAP ESTIMATES")
print("=" * 70)
print(bootstrap_results.round(2).to_string(index=False))

BOOTSTRAP ESTIMATES
state_alpha     crop  estimate  lower_95  upper_95
         CA     CORN     -3.87     -7.40     -0.41
         CA    WHEAT      1.53     -1.12      4.36
         IA     CORN     -0.72     -1.79      0.42
         IA SOYBEANS      0.71     -0.68      2.09
         NE     CORN     -4.13     -6.55     -1.64
         NE  SORGHUM      1.69     -2.90      6.16
         NE SOYBEANS     -1.11     -3.39      1.30
         NE    WHEAT      0.19     -4.04      4.45


## Evidence: Soil water storage does not show a consistent drought-protection effect

**What the evidence does not say:** These results do **not** show that higher soil water storage causes greater or smaller yield losses. For most state-crop combinations, the uncertainty range includes zero, so the data do not provide clear evidence of a difference. The analysis also does not account for all potential factors, such as irrigation and crop productivity.